# BFP知识库构建与检索

为待修复的错误代码片段从知识库中检索语义最相似的BFP

In [ ]:
import sys
import os
from pathlib import Path
# os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

# 向上两级到项目根目录
PROJECT_ROOT = Path(os.path.abspath("")).resolve()
while not (PROJECT_ROOT / "configs").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")


from src.utils.io import (
    load_jsonl, save_json, load_yaml
)
from src.utils.log import (
    setup_logging
)
from src.models.retriever.Base import create_embedder
from src.rag.bfp.milvus_client import BFPMilvusClient
from src.rag.bfp.retriever import BFPRetriever

setup_logging()

## 1. 加载配置

In [ ]:
# db_config = load_yaml(str(PROJECT_ROOT / "configs/database/bfp/white/proxy.yml")) # proxy + test查询集构建
# db_config = load_yaml(str(PROJECT_ROOT / "configs/database/bfp/black/proxy.yml")) # proxy + test查询集构建

db_config = load_yaml(str(PROJECT_ROOT / "configs/database/bfp/target.yml")) # target查询集构建
# db_config = load_yaml(str(PROJECT_ROOT / "configs/database/bfp/black/target.yml")) # target查询集构建


model_config = load_yaml(str(PROJECT_ROOT / db_config["model"]["retriever_config"]))

# 将相对路径转为绝对路径
if not os.path.isabs(db_config["database"]["uri"]):
    db_config["database"]["uri"] = str(PROJECT_ROOT / db_config["database"]["uri"])

print("=== Model Config ===")
for k, v in model_config.items():
    print(f"  {k}: {v}")

print("\n=== Database Config ===")
for k, v in db_config["database"].items():
    print(f"  {k}: {v}")

## 2. 加载检索器

In [ ]:
embedder = create_embedder(model_config)

# 快速验证
test_emb = embedder.embed_queries(["int main() { return 0; }"])
print(f"Test embedding shape: {test_emb.shape}")
print(f"Norm: {(test_emb[0] ** 2).sum() ** 0.5:.4f}")

## 3. 构建 Milvus 知识库
生成 dense embeddings 并连同原始文本一起插入 Milvus，BM25 sparse 向量由 Milvus 内置函数自动生成。

In [ ]:
milvus_client = BFPMilvusClient(db_config)
collection_name = db_config["database"]["collection_name"]
already_built = (
    milvus_client.client.has_collection(collection_name)
    and milvus_client.get_collection_stats()["row_count"] > 0
)

if already_built:
    print("Knowledge base load successfully.")
else:
    # 将相对路径转为绝对路径
    if not os.path.isabs(db_config["data"]["rag_base_file_path"]):
        db_config["data"]["rag_base_file_path"] = str(PROJECT_ROOT / db_config["data"]["rag_base_file_path"])
    
    # 加载数据
    kb_data = load_jsonl(db_config["data"]["rag_base_file_path"])
    print(f"Knowledge base: {len(kb_data)} entries")
    
    # 构建知识库
    milvus_client.build_knowledge_base(embedder, kb_data)
    print("Knowledge base built successfully.")


## 4. 执行检索
对每个查询执行检索

In [ ]:
# 将相对路径转为绝对路径
if not os.path.isabs(db_config["data"]["query_file_path"]):
    db_config["data"]["query_file_path"] = str(PROJECT_ROOT / db_config["data"]["query_file_path"])
query_data = load_jsonl(db_config["data"]["query_file_path"])
print(f"Queries:        {len(query_data)} entries")
print(f"Sample query entry keys: {list(query_data[0].keys())}")

# 加载检索器模块与检索参数
retriever = BFPRetriever(embedder, milvus_client)
top_k = db_config["retrieval"].get("top_k", 10)
retrieval_strategy = db_config["retrieval"].get("retrieval_strategy", "hybrid")
query_target = db_config["retrieval"].get("query_target", "buggy_code")
print(f"Retrieval params: top_k={top_k}, strategy={retrieval_strategy}, query_target={query_target}")

if retrieval_strategy == "dense":
    results = retriever.dense_retrieve(query_data, top_k=top_k, query_target=query_target)
elif retrieval_strategy == "sparse":
    results = retriever.sparse_retrieve(query_data, top_k=top_k, query_target=query_target)
else:  # hybrid
    alpha = db_config["retrieval"].get("alpha_weight", 0.85)
    print(f"  alpha={alpha}")
    results = retriever.hybrid_retrieve(query_data, top_k=top_k, alpha=alpha, query_target=query_target)

print(f"\nRetrieved results for {len(results)} queries")
print(f"Sample: {results[0]['entity']['id']} -> {len(results[0]['retrieval_results'])} hits")
if results[0]["retrieval_results"]:
    hit = results[0]["retrieval_results"][0]
    print(f"  Top hit: doc_id={hit['doc_id']}, score={hit['score']:.4f}")

## 5. 保存检索结果

In [ ]:
# 从 query_file_path 推导保存目录结构
rag_base_file_path_str = str(db_config["data"]["rag_base_file_path"])
query_path_str = str(db_config["data"]["query_file_path"])

# 推导 white/black 子目录
if "black" in query_path_str:
    env_subdir = "black"
else:
    env_subdir = "white"

# 根据检索策略确定 alpha_str
if retrieval_strategy == "dense":
    alpha_str = "1"
elif retrieval_strategy == "sparse":
    alpha_str = "0"
else:  # hybrid
    alpha_str = str(db_config["retrieval"].get("alpha_weight", 0.85))

model_name_without_provider = embedder.get_model_name(with_provider=False)
# 根据 query_file_path 内容确定保存路径
filename = f"num_{len(query_data)}_alpha_{alpha_str}_target_{query_target}_retriever_{model_name_without_provider}_retrieval_results.json"

if "test" in rag_base_file_path_str:
    # test 场景：data/query/{white或black}/test/results/
    output_path = PROJECT_ROOT / f"data/query/{env_subdir}/test/results/{filename}"
elif "vul" in rag_base_file_path_str:
    # vul 场景：data/query/{white或black}/
    output_path = PROJECT_ROOT / f"data/query/{env_subdir}/{filename}"
else:
    # 兜底：保持原逻辑（data/bfp/{subdir}/）
    output_path = PROJECT_ROOT / f"data/bfp/{env_subdir}/{filename}"

# 确保输出目录存在
output_path.parent.mkdir(parents=True, exist_ok=True)

save_json(results, output_path)
print(f"Results saved to {output_path}")

milvus_client.close()
print("Done!")